# Análisis de Cohortes - H&M

Este notebook realiza un análisis profesional de cohortes sobre el dataset de transacciones de H&M. El objetivo es entender la retención y el comportamiento de los clientes a lo largo del tiempo, identificando patrones clave para estrategias de fidelización.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, min as spark_min, month, year, countDistinct
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder.appName("Cohortes_HYM").getOrCreate()

## 1. Carga y exploración inicial de datos

In [ ]:
df = spark.read.parquet('c:/Users/jeffc/Desktop/data-analytics-labs/lab03/merge_pyspark')
print(f"Total de registros: {df.count():,}")
df.printSchema()
df.show(5)

# Estadísticas descriptivas
print("Clientes únicos:", df.select('customer_id').distinct().count())
print("Rango de fechas:", df.selectExpr("min(Fecha) as min_fecha", "max(Fecha) as max_fecha").show())
print("Compras promedio por cliente:", df.groupBy('customer_id').count().agg({'count': 'mean'}).show())

## 2. Distribución temporal de compras

In [ ]:
compras_por_mes = df.groupBy(year('Fecha').alias('año'), month('Fecha').alias('mes')).count().toPandas()
plt.figure(figsize=(10,5))
plt.plot(compras_por_mes['año'].astype(str) + '-' + compras_por_mes['mes'].astype(str), compras_por_mes['count'], marker='o')
plt.xticks(rotation=45)
plt.title('Distribución temporal de compras')
plt.xlabel('Mes')
plt.ylabel('Número de compras')
plt.tight_layout()
plt.show()

## 3. Limpieza y preparación para análisis de cohortes

In [ ]:
df_clean = df.select('customer_id', 'Fecha', 'price') \
    .where(col('customer_id').isNotNull() & col('Fecha').isNotNull() & (col('price') > 0))
print(f"Registros limpios: {df_clean.count():,}")

## 4. Construcción de cohortes mensuales

In [ ]:
cohort_df = df_clean.groupBy('customer_id').agg(spark_min('Fecha').alias('cohort_date'))
cohort_df = cohort_df.withColumn('cohort_year', year('cohort_date'))
cohort_df = cohort_df.withColumn('cohort_month', month('cohort_date'))

df_cohort = df_clean.join(cohort_df, on='customer_id')
df_cohort = df_cohort.withColumn('trans_year', year('Fecha'))
df_cohort = df_cohort.withColumn('trans_month', month('Fecha'))

## 5. Cálculo del índice de cohorte (meses desde primera compra)

In [ ]:
df_cohort = df_cohort.withColumn(
    'cohort_index',
    (col('trans_year') - col('cohort_year')) * 12 + (col('trans_month') - col('cohort_month')) + 1
)

## 6. Matriz de retención de cohortes

In [ ]:
retention = df_cohort.groupBy('cohort_year', 'cohort_month', 'cohort_index') \
    .agg(countDistinct('customer_id').alias('num_customers'))
retention_pd = retention.toPandas()
cohort_pivot = retention_pd.pivot_table(
    index=['cohort_year', 'cohort_month'],
    columns='cohort_index',
    values='num_customers'
)
cohort_size = cohort_pivot.iloc[:, 0]
retention_rate = cohort_pivot.divide(cohort_size, axis=0)

## 7. Visualización de la matriz de retención de cohortes

In [ ]:
plt.figure(figsize=(14, 8))
sns.heatmap(retention_rate, annot=True, fmt='.0%', cmap='Blues')
plt.title('Matriz de Retención de Cohortes (H&M)', fontsize=18)
plt.xlabel('Mes desde primera compra')
plt.ylabel('Cohorte (Año, Mes)')
plt.show()

## 8. Análisis de cohortes y recomendaciones

In [ ]:
print(f"Cohortes analizadas: {len(retention_rate)}")
print(f"Promedio de retención al mes 1: {retention_rate[1].mean():.2%}")
print(f"Promedio de retención al mes 3: {retention_rate[3].mean():.2%}")
if 6 in retention_rate.columns:
    promedio_mes_6 = f"{retention_rate[6].mean():.2%}"
else:
    promedio_mes_6 = "No disponible"
print(f"Promedio de retención al mes 6: {promedio_mes_6}")

print('''
Recomendaciones:
- Identificar cohortes con mayor caída de retención y analizar causas.
- Implementar campañas de fidelización en los primeros 3 meses.
- Personalizar comunicaciones para cohortes con mejor retención.
- Realizar seguimiento a cohortes nuevas para evaluar impacto de acciones.
''')

## 9. Exportación de resultados y cierre de Spark

In [ ]:
retention_rate.to_csv('matriz_retencion_cohortes_hym.csv')
print('Matriz de retención exportada a matriz_retencion_cohortes_hym.csv')
spark.stop()